# Scanned PDF → Markdown on a Colab GPU

Runs the [marker](https://github.com/datalab-to/marker)-based pipeline from
[`pdf-to-markdown-marker`](https://github.com/donjoe1996/pdf-to-markdown-marker), tuned for a 1962 scan of *Being and
Time* that is stored as two-page spreads.

**Why a GPU.** On an Apple M2 the run takes ~7 hours: the surya vision model
generates ~1,500 tokens per page, and each token re-reads the whole 1.3 GB model
from memory, so the work is memory-bandwidth bound. A T4 has roughly 3x the
bandwidth *and* enough spare VRAM to OCR several pages at once.

**Two Colab-specific obstacles, both handled below:**

1. marker's default NVIDIA backend spawns a **Docker** image, and Colab has no
   Docker daemon. We force the `llamacpp` backend, which offloads every layer to
   CUDA via `-ngl 99` — the same path already validated on the Mac.
2. llama.cpp ships prebuilt CUDA binaries **for Windows only**, so
   `llama-server` is compiled here (~5–10 min) and **cached to Drive** so later
   sessions skip the build.

Run the cells in order.

## 1. Confirm a GPU runtime

If this shows no GPU: **Runtime → Change runtime type → T4 GPU**, then rerun.

In [ ]:
!nvidia-smi || echo "NO GPU -- set Runtime > Change runtime type > T4 GPU"

## 2. Get the code

Cloned fresh each session. Rerunning this cell pulls the latest version.

In [ ]:
REPO = 'https://github.com/donjoe1996/pdf-to-markdown-marker.git'
CODE = '/content/pdf-to-markdown-marker'

import os, shutil, sys
if os.path.isdir(CODE):
    shutil.rmtree(CODE)          # always start from a clean checkout
!git clone --depth 1 $REPO $CODE

sys.path.insert(0, CODE)
os.chdir(CODE)
print('code:', CODE)

## 3. Mount Drive

Drive holds the two things that must outlive a session: **the source PDF** (too
large for git, and not ours to redistribute) and **the output**, so a
disconnect resumes rather than restarting. The compiled `llama-server` is
cached there too.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Point at your PDF

Upload the scanned PDF to Drive and set `PDF` to its path. Everything else is
derived. `WORK` stays on local disk because marker writes scratch files and
Drive I/O is slow.

In [ ]:
DRIVE  = '/content/drive/MyDrive/being-and-time'   # your folder in Drive
PDF    = f'{DRIVE}/42700894-Martin-Heidegger-Being-and-Time.pdf'
OUTDIR = f'{DRIVE}/output'                         # resumable: keep on Drive
CACHE  = f'{DRIVE}/llama-cache'                    # compiled llama-server

import os
os.makedirs(DRIVE, exist_ok=True)
os.makedirs(OUTDIR, exist_ok=True)
assert os.path.isfile(PDF), (
    f'PDF not found: {PDF}\n'
    'Upload it to that folder in Drive, or edit DRIVE/PDF above.'
)
print('pdf   :', PDF)
print('output:', OUTDIR)

## 5. Install dependencies

Colab already ships a CUDA build of torch, and marker needs `torch>=2.7,<3`. If
the preinstalled one satisfies that, pip leaves it alone — do **not** force a
reinstall, or you risk replacing a working CUDA torch.

In [ ]:
!pip install -q marker-pdf pymupdf

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0))

## 6. Build the CUDA `llama-server`

First run compiles it (~5–10 min) and caches it to Drive; later sessions reuse
the cached binary in seconds. Concurrency is sized from detected VRAM — a 16 GB
T4 takes 4 pages at once.

In [ ]:
from bt import gpu_setup

gpu_setup.report()
env = gpu_setup.configure(cache_dir=CACHE)   # builds if not cached, sets env vars

## 7. Download the models (~1.8 GB)

Fetched up front deliberately: marker starts each model in its own server
subprocess with a 300s health check, and on a cold cache the download outlives
that check, so marker force-kills its own server with a confusing `SpawnError`.

The previous cell also set `HF_HUB_DISABLE_XET=1`; without it Hub downloads can
hang at 0 bytes indefinitely on some networks.

In [ ]:
from bt.warmup import warm_all
warm_all()

## 8. Split the two-page spreads

Each PDF page is a scanned spread of two facing book pages. They are cut apart
first, at a gutter detected per page — it drifts between 0.497 and 0.514, so a
fixed 50% split would clip text.

Expect **582** pages from 294 spreads; six halves are blank.

In [ ]:
from pathlib import Path
from bt.split_spreads import split_document

split_pdf = Path(OUTDIR) / 'pages.pdf'
records = split_document(Path(PDF), split_pdf, Path(OUTDIR) / 'pagemap.json')
print(f'{len(records)} book pages -> {split_pdf}')

## 9. Smoke test — two pages (recommended)

Page 0 is the opening page: dense polytonic Greek plus a half-page footnote
block, the hardest content in the book.

The check that matters is **fresh OCR**. The PDF carries a bad Acrobat OCR
layer, and if marker ever reads that instead of OCRing you get readable Markdown
made of the wrong characters. `verify` greps for that layer's distinctive damage.

In [ ]:
import time
from bt.transcribe import transcribe
from bt.verify import run_all

t0 = time.time()
transcribe(split_pdf, Path('/content/sample.md'),
           Path('/content/chunks-sample'), total_pages=2, chunk_size=2)
print(f'--- {(time.time()-t0)/2:.1f}s per page ---')

for f in run_all(Path('/content/sample.md').read_text()):
    print(f"[{'ok  ' if f.ok else 'FAIL'}] {f.name:<13} {f.detail}")

## 10. Transcribe the whole book

Chunks are written to Drive as they finish, so this is **resumable**: if Colab
disconnects, just rerun this cell and it skips what is already done.

Free-tier sessions idle out after ~90 minutes, so keep the tab active.

In [ ]:
raw_md = Path(OUTDIR) / 'raw.md'
transcribe(
    split_pdf, raw_md,
    Path(OUTDIR) / 'chunks',
    total_pages=len(records),
    chunk_size=10,        # smaller chunks = finer-grained resume
    dpi=300,              # matches the scan; lowering it does NOT speed things up
)

## 11. Clean up and verify

Strips running heads, namespaces footnote ids per page (footnote "1" recurs on
nearly every page, so un-namespaced ids would collide hundreds of times), and
rejoins words hyphenated across line breaks.

In [ ]:
from bt.postprocess import process

final_md = Path(OUTDIR) / 'being-and-time.md'
cleaned, stats = process(raw_md.read_text(encoding='utf-8'))
final_md.write_text(cleaned, encoding='utf-8')
print(stats.render())

print()
for f in run_all(cleaned):
    print(f"[{'ok  ' if f.ok else 'FAIL'}] {f.name:<13} {f.detail}")
print(f'\nWrote {final_md} ({len(cleaned):,} chars)')

## 12. Download

Already saved to Drive; this is just a direct download.

In [ ]:
from google.colab import files
files.download(str(final_md))

---

### Troubleshooting

| Symptom | Cause / fix |
|---|---|
| `No CUDA GPU visible` | Runtime → Change runtime type → T4 GPU |
| Build fails, `nvcc not found` | Not on a GPU runtime |
| `SpawnError: ... failed to become healthy` | Models weren't cached — rerun cell 7 |
| Hub download stuck at 0 bytes | `HF_HUB_DISABLE_XET=1` missing — rerun cell 6 |
| Session disconnected | Rerun cell 10; completed chunks are skipped |
| Out of VRAM | `gpu_setup.configure(cache_dir=CACHE, parallel=2)` |
| `PDF not found` | Upload the scan to the `DRIVE` folder in cell 4 |

Lowering `dpi` is **not** a speed fix: 192 vs 300 DPI measured 92.1 vs
82.5–95.2 s/page on the Mac for 99.71% identical output. The cost is dominated
by tokens generated, not pixels read.